# Aerial Object Detection using HerdNet
https://github.com/Alexandre-Delplanque/HerdNet


## Setup

This notebook requires the **fit-training** conda environment which
includes `animaloc` (HerdNet). See `INSTALLATION_INSTRUCTIONS.md`.

```bash
conda activate fit-training
```

In [1]:
# Colab only — install dependencies if not already available
# import sys

# !git clone -b course_draft https://github.com/cwinkelmann/usde-innovations-applications-forest-it.git fit-course
# !cd fit-course && git pull
# !pip install -e "./fit-course[herdnet,dev]"
#
# sys.path.append('./fit-course')

In [2]:
import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

Using device: mps


In [3]:
# configure the logger
from loguru import logger

logger.disable("animaloc")


In [4]:
# Verify animaloc is installed (comes with fit-training env)
import animaloc
print(f"animaloc {animaloc.__version__}")

/Users/christian/opt/anaconda3/envs/fit-training/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


animaloc 0.2.1


Load a seperate Sample of the test set

In [5]:
from huggingface_hub import hf_hub_download, snapshot_download
from pathlib import Path

local_dir = Path("./models").resolve()

# Download model weights and config
model_path = hf_hub_download(
    repo_id="karisu/HerdNet",
    filename="general_2022/20220413_HerdNet_General_dataset_2022.pth",
    local_dir=local_dir
)
config_path = hf_hub_download(
    repo_id="karisu/HerdNet",
    filename="general_2022/config.yaml",
    local_dir=local_dir
)
print(f"Model: {model_path}\nConfig: {config_path}")

Model: /Users/christian/work/hnee/usde-innovations-applications-forest-it/week1/practicals/models/general_2022/20220413_HerdNet_General_dataset_2022.pth
Config: /Users/christian/work/hnee/usde-innovations-applications-forest-it/week1/practicals/models/general_2022/config.yaml


Load a part of the General Dataset which is hosted on Huggingface
https://dataverse.uliege.be/dataset.xhtml?persistentId=doi:10.58119/ULG/MIRUU5


In [6]:

# Download a small sample of the test set
sample_data_path = snapshot_download(
    repo_id="karisu/General_Dataset",
    repo_type="dataset",
    local_dir="./data",
    allow_patterns=["test_sample/*"],
    revision="main",
)
test_sample_annotation_path = hf_hub_download(
    repo_id="karisu/General_Dataset",
    repo_type="dataset",
    filename="test_sample.csv",
    local_dir="./data",
)
print(f"Test data: {sample_data_path}")

Fetching ... files: 7it [00:02,  3.19it/s]


Test data: /Users/christian/work/hnee/usde-innovations-applications-forest-it/week1/practicals/data


In [7]:
# Training, validation and test datasets
import albumentations as A

from animaloc.datasets import CSVDataset
from animaloc.data.transforms import MultiTransformsWrapper, DownSample, PointsToMask, FIDT

patch_size = 512
num_classes = 7
down_ratio = 2

test_dataset = CSVDataset(
    csv_file = './data/test_sample.csv',
    root_dir = './data/test_sample',
    albu_transforms = [A.Normalize(p=1.0)],
    end_transforms = [DownSample(down_ratio=down_ratio, anno_type='point')]
    )

In [8]:
# Test installation
# Set the seed
from animaloc.utils.seed import set_seed
set_seed(9292)


In [9]:
# Dataloaders
from torch.utils.data import DataLoader

test_dataloader = DataLoader(dataset = test_dataset, batch_size = 1, shuffle = False)

The Loss Wrapper is needed to evaluate the model against a known test-set

In [10]:
from animaloc.models import HerdNet
from torch import Tensor
from animaloc.models import LossWrapper
from animaloc.train.losses import FocalLoss
from torch.nn import CrossEntropyLoss

herdnet = HerdNet(num_classes=num_classes, down_ratio=down_ratio).to(device)

weight = Tensor([0.1, 1.0, 2.0, 1.0, 6.0, 12.0, 1.0]).to(device)

losses = [
    {'loss': FocalLoss(reduction='mean'), 'idx': 0, 'idy': 0, 'lambda': 1.0, 'name': 'focal_loss'},
    {'loss': CrossEntropyLoss(reduction='mean', weight=weight), 'idx': 1, 'idy': 1, 'lambda': 1.0, 'name': 'ce_loss'}
    ]

## This loss Wrapper is important for the Evaluator even if there will be no Training
herdnet = LossWrapper(herdnet, losses=losses)

In [11]:
from torch.optim import Adam
from animaloc.train import Trainer
from animaloc.eval import PointsMetrics, HerdNetStitcher, HerdNetEvaluator
from animaloc.utils.useful_funcs import mkdir

work_dir = './output'
mkdir(work_dir)

metrics = PointsMetrics(radius=20, num_classes=num_classes)

stitcher = HerdNetStitcher(
    model=herdnet,
    size=(patch_size, patch_size),
    overlap=160,
    down_ratio=down_ratio,
    reduction='mean',
    device_name=str(device),
)

## Running HerdNet

At first a pretrained model is run against the Test Subset on African Megafauna.
Then it shown how to train the model.


### Test the pretrained model

A downloaded model is run against a small data.

In [12]:

# Load trained parameters
from animaloc.models import load_model
pth_path = './models/general_2022/20220413_HerdNet_General_dataset_2022.pth'

herdnet = load_model(herdnet, pth_path=pth_path)
herdnet = LossWrapper(herdnet, losses=losses)

In [13]:
# Create output folder
test_dir = './test_output'
mkdir(test_dir)

In [14]:
# Create an Evaluator
test_evaluator = HerdNetEvaluator(
    model=herdnet,
    dataloader=test_dataloader,
    metrics=metrics,
    stitcher=stitcher,
    work_dir=test_dir,
    header='test',
    device_name=str(device),
    )

In [15]:
# Start testing
test_f1_score = test_evaluator.evaluate(returns='f1_score')

/Users/christian/opt/anaconda3/envs/fit-training/lib/python3.11/site-packages/albumentations/core/composition.py:331: UserWarning: Got processor for keypoints, but no transform to process it.
  self._set_keys()


test [1/7] eta: 0:01:02 n: 11 tp: 1 fp: 2 fn: 8 recall: 0.11 precision: 0.33 f1_score: 0.17 f2_score: 0.13 f5_score: 0.11 MAE: 6.0 ME: 0.0 MSE: 36.0 RMSE: 6.0 avg_score: 0.01 avg_dscore: 0.084 time: 8.9384 data: 0.3645


/Users/christian/opt/anaconda3/envs/fit-training/lib/python3.11/site-packages/albumentations/core/composition.py:331: UserWarning: Got processor for keypoints, but no transform to process it.
  self._set_keys()
/Users/christian/opt/anaconda3/envs/fit-training/lib/python3.11/site-packages/albumentations/core/composition.py:331: UserWarning: Got processor for keypoints, but no transform to process it.
  self._set_keys()
/Users/christian/opt/anaconda3/envs/fit-training/lib/python3.11/site-packages/albumentations/core/composition.py:331: UserWarning: Got processor for keypoints, but no transform to process it.
  self._set_keys()
/Users/christian/opt/anaconda3/envs/fit-training/lib/python3.11/site-packages/albumentations/core/composition.py:331: UserWarning: Got processor for keypoints, but no transform to process it.
  self._set_keys()
/Users/christian/opt/anaconda3/envs/fit-training/lib/python3.11/site-packages/albumentations/core/composition.py:331: UserWarning: Got processor for keypoin

test [7/7] eta: 0:00:11 n: 13 tp: 7 fp: 1 fn: 5 recall: 0.58 precision: 0.88 f1_score: 0.7 f2_score: 0.63 f5_score: 0.59 MAE: 4.0 ME: 0.0 MSE: 16.0 RMSE: 4.0 avg_score: 0.83 avg_dscore: 0.22 time: 11.7918 data: 0.5457
test Total time: 0:01:22 (11.7929 s / it)


In [16]:
# Print global F1 score (%)
print(f"F1 score = {test_f1_score * 100:0.0f}%")


F1 score = 62%


In [17]:
# Get the detections
detections = test_evaluator.results
detections

,class,n,recall,precision,f1_score,confusion,mae,me,mse,rmse,ap
0,1,0,0.000000,0.00000,0.000000,1.0,0.000000,0.000000,0.000000,0.0000,0.000000
1,2,0,0.000000,0.00000,0.000000,1.0,0.000000,0.000000,0.000000,0.0000,0.000000
2,3,0,0.000000,0.00000,0.000000,1.0,0.000000,0.000000,0.000000,0.0000,0.000000
3,4,0,0.000000,0.00000,0.000000,1.0,0.000000,0.000000,0.000000,0.0000,0.000000
4,5,0,0.000000,0.00000,0.000000,1.0,0.000000,0.000000,0.000000,0.0000,0.000000
5,6,41,0.560976,0.69697,0.621622,0.0,2.285714,-1.142857,8.571429,2.9277,0.530441
6,binary,41,0.560976,0.69697,0.621622,0.0,2.285714,0.000000,8.571429,2.9277,0.530441


## Inferencing using a config file
Here just the model and configuration is loaded making the process transferable. Any config can be overridden.

In [18]:

from pathlib import Path
from hydra import initialize_config_dir, compose
from hydra.core.global_hydra import GlobalHydra
from omegaconf import OmegaConf

# Clear any previous Hydra state (important in notebooks when re-running cells)
GlobalHydra.instance().clear()

# Configure paths
config_dir = str(Path.cwd() / "models/general_2022")  # adjust as needed
config_name = "config"

# Load config
with initialize_config_dir(config_dir=config_dir, version_base="1.1"):
    cfg = compose(config_name=config_name, overrides=[
        # Add any overrides here, e.g.:
        # "training.epochs=10",
        #  "training.batch_size=4",
        "model.load_from=./models/general_2022/20220413_HerdNet_General_dataset_2022.pth",
        "datasets.test.root_dir=./data/test_sample/"
    ])

# Inspect config (optional) It contains all which was used for training
print(OmegaConf.to_yaml(cfg))

losses:
  FocalLoss:
    print_name: focal_loss
    from_torch: false
    output_idx: 0
    target_idx: 0
    lambda_const: 1.0
    kwargs:
      alpha: 2
      beta: 4
      reduction: mean
      normalize: false
  CrossEntropyLoss:
    print_name: ce_loss
    from_torch: true
    output_idx: 1
    target_idx: 1
    lambda_const: 1.0
    background_class_weight: 0.1
    kwargs:
      reduction: mean
      weight:
      - ${losses.CrossEntropyLoss.background_class_weight}
      - 1.0
      - 2.0
      - 1.0
      - 6.0
      - 12.0
      - 1.0
datasets:
  img_size:
  - 512
  - 512
  anno_type: point
  num_classes: 7
  collate_fn: null
  class_def:
    1: Alcelaphinae
    2: Buffalo
    3: Kob
    4: Warthog
    5: Waterbuck
    6: Elephant
  train:
    name: CSVDataset
    csv_file: /raid/cwinkelmann/training_data/iguana/2025_12_02/last_run/train/herdnet_format_800_160_crops.csv
    root_dir: /raid/cwinkelmann/training_data/iguana/2025_12_02/last_run/train/crops_800_numNone_overlap160


In [19]:
from animaloc.utils.inference import inference

# Run inference
detections = inference(cfg, plain_inference=True, vis_detections=True)



/Users/christian/opt/anaconda3/envs/fit-training/lib/python3.11/site-packages/albumentations/core/composition.py:331: UserWarning: Got processor for keypoints, but no transform to process it.
  self._set_keys()


[TEST] [1/7] eta: 0:34:36 n: 12 tp: 0 fp: 11 fn: 1 recall: 0.0 precision: 0.0 f1_score: 0.0 f2_score: 0.0 f5_score: 0.0 MAE: 10.0 ME: -1.0 MSE: 100.0 RMSE: 10.0 avg_score: 0.73 avg_dscore: 0.204 time: 296.5845 data: 0.6458


/Users/christian/opt/anaconda3/envs/fit-training/lib/python3.11/site-packages/albumentations/core/composition.py:331: UserWarning: Got processor for keypoints, but no transform to process it.
  self._set_keys()
/Users/christian/opt/anaconda3/envs/fit-training/lib/python3.11/site-packages/albumentations/core/composition.py:331: UserWarning: Got processor for keypoints, but no transform to process it.
  self._set_keys()
/Users/christian/opt/anaconda3/envs/fit-training/lib/python3.11/site-packages/albumentations/core/composition.py:331: UserWarning: Got processor for keypoints, but no transform to process it.
  self._set_keys()
/Users/christian/opt/anaconda3/envs/fit-training/lib/python3.11/site-packages/albumentations/core/composition.py:331: UserWarning: Got processor for keypoints, but no transform to process it.
  self._set_keys()
/Users/christian/opt/anaconda3/envs/fit-training/lib/python3.11/site-packages/albumentations/core/composition.py:331: UserWarning: Got processor for keypoin

KeyboardInterrupt: 

In [ ]:
# Show results
print(f"Total detections: {len(detections)}")
detections.head(10)

### Visualise detections

In [ ]:
%matplotlib inline

import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

# Load data
gt_df = pd.read_csv("data/test_sample.csv")
det_df = pd.read_csv("detections.csv")

# First image
image_stem = "02033bf9b6c41f5815072434f8d61707cc8ea1fb"
image_path = f"data/test_sample/{image_stem}.jpeg"

# Get ground truth annotations (handle .JPG extension in CSV)
gt = gt_df[gt_df["images"].str.startswith(image_stem)]

# Get detections
det = det_df[det_df["images"] == f"{image_stem}.jpeg"].dropna(subset=["x", "y"])

# Plot
img = Image.open(image_path)
fig, ax = plt.subplots(figsize=(14, 10))
ax.imshow(img)

# Ground truth in green
ax.scatter(gt["x"], gt["y"], c="lime", s=100, marker="o", 
           edgecolors="black", linewidths=1.5, label=f"Ground Truth ({len(gt)})")

# Detections in red
ax.scatter(det["x"], det["y"], c="red", s=100, marker="x", 
           linewidths=2, label=f"Detections ({len(det)})")

ax.set_title(f"{image_stem}\nGT: {len(gt)} | Det: {len(det)}")
ax.legend(loc="upper right")
ax.axis("off")
plt.tight_layout()
plt.savefig("detection_plot.png", dpi=150, bbox_inches="tight")


#### Looking into single predictions
Those are with a reasonably high confidence Score.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

# Image stem to plot
thumbnails_folder = Path("thumbnails")

# Find all thumbnails for this stem
thumbnails = sorted(thumbnails_folder.glob(f"{image_stem}._*.JPG"))

print(f"Found {len(thumbnails)} thumbnails")

# Plot all in a grid
n_cols = 4
n_rows = (len(thumbnails) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 4 * n_rows))
axes = axes.flatten() if len(thumbnails) > 1 else [axes]

for ax, thumb_path in zip(axes, thumbnails):
    img = Image.open(thumb_path)
    ax.imshow(img)
    ax.set_title(thumb_path.name, fontsize=8)
    ax.axis("off")

# Hide empty subplots
for ax in axes[len(thumbnails):]:
    ax.axis("off")

plt.suptitle(f"{image_stem} ({len(thumbnails)} thumbnails)", fontsize=12)
plt.tight_layout()
plt.savefig(f"{image_stem}_thumbnails.png", dpi=150, bbox_inches="tight")

### Manual Patch creation
Patch larger images into trainable tiles

In [ ]:
# Create validation patches using the Patcher API
from animaloc.data.patches import Patcher

if not Path("./data/val_patches").exists():
    mkdir('./data/val_patches')
    patcher = Patcher(
        image_folder='./data/val',
        annot_file='./data/val.csv',
        patch_size=(512, 512),
        overlap=0,
        output_folder='./data/val_patches',
        min_visibility=0.0,
        keep_all=False
    )
    patcher.patch()
else:
    print("Validation Patches already exist")

In [ ]:
train_dataloader = DataLoader(dataset = train_dataset, batch_size = 12, shuffle = True)

val_dataloader = DataLoader(dataset = val_dataset, batch_size = 1, shuffle = False)

In [ ]:
lr = 1e-5
weight_decay = 1e-3
epochs = 2

optimizer = Adam(params=herdnet.parameters(), lr=lr, weight_decay=weight_decay)

In [ ]:
trainer = Trainer(
    model=herdnet,
    train_dataloader=train_dataloader,
    optimizer=optimizer,
    num_epochs=epochs,
    evaluator=evaluator,
    work_dir=work_dir
    )

### Training
(about 15 minutes per epoch on RTX 4090 )

In [ ]:
# Start the training, the Focal and Cross Entropy Loss should already be very low from the start.
trainer.start(warmup_iters=100, checkpoints='best', select='max', validate_on='f1_score')


### Evaluate the just trained model

In [ ]:
# Load trained parameters
from animaloc.models import load_model
pth_path = './output/best_model.pth'

herdnet_trained = load_model(herdnet, pth_path=pth_path)
herdnet_trained = LossWrapper(herdnet, losses=losses)

In [ ]:
test_dataset = CSVDataset(
    csv_file = './data/test_sample.csv',
    root_dir = './data/test_sample',
    albu_transforms = [A.Normalize(p=1.0)],
    end_transforms = [DownSample(down_ratio=down_ratio, anno_type='point')]
    )


test_dataloader = DataLoader(dataset = test_dataset, batch_size = 1, shuffle = False)

In [ ]:
# Create an Evaluator
test_evaluator = HerdNetEvaluator(
    model=herdnet_trained,
    dataloader=test_dataloader,
    metrics=metrics,
    stitcher=stitcher,
    work_dir=test_dir,
    header='test'
    )

In [ ]:
# Print global F1 score (%)
test_f1_score = test_evaluator.evaluate(returns='f1_score')
print(f"F1 score = {test_f1_score * 100:0.0f}%")

### Reproducable Training based on a Config
This training and inference can be based on the same configuration. This reduces the setup.

In [ ]:
from animaloc.utils.train import main

# Clear any previous Hydra state (important in notebooks when re-running cells)
GlobalHydra.instance().clear()

# Configure paths
config_dir = str(Path("models/general_2022/").resolve())  # adjust as needed
config_name = "config"

# Load config
with initialize_config_dir(config_dir=config_dir, version_base="1.1"):
    cfg = compose(config_name=config_name, overrides=[
        # Add any overrides here, e.g.:
        "training_settings.epochs=1",
        "training_settings.batch_size=24",
        "model.load_from=models/general_2022/20220413_HerdNet_General_dataset_2022.pth",
        "datasets.train.csv_file=data/train_patches.csv",
        "datasets.train.root_dir=data/train_patches",
        "datasets.validate.csv_file=data/val_patches/gt.csv",
        "datasets.validate.root_dir=data/val_patches",
        "wandb_flag=true",
    ])

# Inspect config (optional)
# print(OmegaConf.to_yaml(cfg))

In [ ]:
# Run training using 
result = main(cfg)

### Visualisation
#### Training Data
![Augmented Example](./visualizations/augmented_dataset_example_3.png)

#### Heatmap on Validation
![Heatmap](./visualizations/heatmap_overlay_005c952ba7a612c40986806cc84a87e1573ef4f2_14.JPG_1.png)

#### Heatmap and Ground Truth
![Heatmap](./visualizations/heatmap_target_005c952ba7a612c40986806cc84a87e1573ef4f2_14.JPG_1.png)